# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)

/Users/meghangillen/ai/projects/tinyml-arduino/bin/python


In [2]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"

In [3]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [5]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [6]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

# <-- Enter your code here <--#
X = df.drop(columns=["Class"])
y = df["Class"]

In [7]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

# <-- Enter your code here <--#
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

In [8]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

# <-- Enter your code here <--#
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

# <-- Enter your code here <--#
y_train_OHE = tf.keras.utils.to_categorical(y_train, num_classes=num_classes)
y_test_OHE = tf.keras.utils.to_categorical(y_test, num_classes=num_classes)

In [10]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

# <-- Enter your code here <--#

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(num_features,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

In [11]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

# <-- Enter your code here <--#
model.compile(
    optimizer=tf.keras.optimizers.legacy.Adam(learning_rate=0.005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train,
    y_train_OHE,
    epochs=20,
    batch_size=8,
    validation_split=0.2
)

Epoch 1/20
13/13 [==============================] - 0s 6ms/step - loss: 63.7503 - accuracy: 0.3232 - val_loss: 27.9954 - val_accuracy: 0.4400
Epoch 2/20
13/13 [==============================] - 0s 1ms/step - loss: 13.7555 - accuracy: 0.4545 - val_loss: 1.6851 - val_accuracy: 0.6400
Epoch 3/20
13/13 [==============================] - 0s 1ms/step - loss: 3.2430 - accuracy: 0.6263 - val_loss: 1.9030 - val_accuracy: 0.4800
Epoch 4/20
13/13 [==============================] - 0s 1ms/step - loss: 1.4067 - accuracy: 0.6263 - val_loss: 1.5025 - val_accuracy: 0.6000
Epoch 5/20
13/13 [==============================] - 0s 1ms/step - loss: 0.9644 - accuracy: 0.7071 - val_loss: 1.0383 - val_accuracy: 0.6000
Epoch 6/20
13/13 [==============================] - 0s 1ms/step - loss: 0.9701 - accuracy: 0.6970 - val_loss: 1.7654 - val_accuracy: 0.4800
Epoch 7/20
13/13 [==============================] - 0s 1ms/step - loss: 1.6887 - accuracy: 0.5657 - val_loss: 2.5384 - val_accuracy: 0.4800
Epoch 8/20
13/13 

In [12]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

# <-- Enter your code here <--#

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y_pred = model.predict(X_test)
y_pred_classes = y_pred.argmax(axis=1)
y_true = y_test_OHE.argmax(axis=1)

print("Accuracy:", accuracy_score(y_true, y_pred_classes))
print(classification_report(y_true, y_pred_classes))
print(confusion_matrix(y_true, y_pred_classes))


2/2 [==============================] - 0s 982us/step
Accuracy: 0.8333333333333334
              precision    recall  f1-score   support

           0       1.00      0.84      0.91        19
           1       0.70      1.00      0.82        21
           2       1.00      0.57      0.73        14

    accuracy                           0.83        54
   macro avg       0.90      0.80      0.82        54
weighted avg       0.88      0.83      0.83        54

[[16  3  0]
 [ 0 21  0]
 [ 0  6  8]]


In [13]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

# <-- Enter your code here <--#
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the model
with open("model_base.tflite", "wb") as f:
    f.write(tflite_model)

# Print file size in KB
import os

file_size_kb = os.path.getsize("model_base.tflite") / 1024
print(f"TFLite model size: {file_size_kb:.2f} KB")

INFO:tensorflow:Assets written to: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmpwggcjop_/assets


INFO:tensorflow:Assets written to: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmpwggcjop_/assets


TFLite model size: 14.06 KB


2026-05-20 11:01:12.450142: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 11:01:12.450162: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 11:01:12.450341: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmpwggcjop_
2026-05-20 11:01:12.450730: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 11:01:12.450734: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmpwggcjop_
2026-05-20 11:01:12.451542: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:382] MLIR V1 optimization pass is not enabled
2026-05-20 11:01:12.451945: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 11:01:12.470959: I tensorflow/cc/saved_model/loader.

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [14]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        # (b) Provide representative_data_gen(X_train_scaled).
        converter.representative_dataset = lambda: representative_data_gen(X_train_scaled)        
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        # (d) Set inference_input_type and inference_output_type to tf.int8.
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8
        
        # <-- Enter your code here <--#
        #pass

    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        # (b) Set supported_types to [tf.float16].
        converter.target_spec.supported_types = [tf.float16]
        # <-- Enter your code here <--#
        #pass

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

        # <-- Enter your code here <--#
        #pass

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.
    tflite_model = converter.convert()

    with open(filename, "wb") as f:
        f.write(tflite_model)

    # <-- Enter your code here <--#
    
    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).

    # <-- Enter your code here for TFLite inference <--#
    interpreter = tf.lite.Interpreter(model_content=tflite_model)

    interpreter.allocate_tensors()
    
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    input_scale, input_zero_point = input_details[0]['quantization']
    
    output_scale, output_zero_point = output_details[0]['quantization']

    y_pred = []

    for i in range(len(X_test)):
        x = X_test[i:i+1].astype(np.float32)

        if input_details[0]['dtype'] == np.int8:
            x = x / input_scale + input_zero_point
            x = np.round(x).astype(np.int8)

        interpreter.set_tensor(input_details[0]['index'], x)
        interpreter.invoke()

        output = interpreter.get_tensor(output_details[0]['index'])

        if output_details[0]['dtype'] == np.int8:
            output = (output - output_zero_point) * output_scale

        y_pred.append(np.argmax(output))

    y_true = np.argmax(y_test_cat, axis=1)

    # Step 4: Report results.
    file_size_kb = os.path.getsize(filename) / 1024
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb:.2f} KB")

    # <-- Enter your code here: print classification_report and confusion_matrix <--#
    print(classification_report(y_true, y_pred))
    print(confusion_matrix(y_true, y_pred))

In [15]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

# <-- Enter your code here <--#

quantize_and_evaluate(model, X_test, y_test_OHE, 'int8', 'model_int8.tflite')
quantize_and_evaluate(model, X_test, y_test_OHE, 'float16', 'model_float16.tflite')
quantize_and_evaluate(model, X_test, y_test_OHE, 'dynamic', 'model_dynamic.tflite')

INFO:tensorflow:Assets written to: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmpv6c92e_t/assets


INFO:tensorflow:Assets written to: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmpv6c92e_t/assets
/Users/meghangillen/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-20 11:01:12.828716: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 11:01:12.828731: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 11:01:12.828866: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmpv6c92e_t
2026-05-20 11:01:12.829258: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 11:01:12.829263: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/ft/hk0gywbs


INT8 TFLite model size: 5.73 KB
              precision    recall  f1-score   support

           0       1.00      0.11      0.19        19
           1       0.38      0.14      0.21        21
           2       0.20      0.64      0.31        14

    accuracy                           0.26        54
   macro avg       0.53      0.30      0.24        54
weighted avg       0.55      0.26      0.23        54

[[ 2  0 17]
 [ 0  3 18]
 [ 0  5  9]]
INFO:tensorflow:Assets written to: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmprrxljfiz/assets


fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
INFO:tensorflow:Assets written to: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmprrxljfiz/assets



FLOAT16 TFLite model size: 8.94 KB
              precision    recall  f1-score   support

           0       1.00      0.84      0.91        19
           1       0.70      1.00      0.82        21
           2       1.00      0.57      0.73        14

    accuracy                           0.83        54
   macro avg       0.90      0.80      0.82        54
weighted avg       0.88      0.83      0.83        54

[[16  3  0]
 [ 0 21  0]
 [ 0  6  8]]
INFO:tensorflow:Assets written to: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmp37f9xys1/assets


2026-05-20 11:01:13.233016: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 11:01:13.233028: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 11:01:13.233133: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmprrxljfiz
2026-05-20 11:01:13.233553: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 11:01:13.233558: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmprrxljfiz
2026-05-20 11:01:13.234752: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 11:01:13.254652: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmprrxljfiz
2026-05-


DYNAMIC TFLite model size: 8.16 KB
              precision    recall  f1-score   support

           0       1.00      0.84      0.91        19
           1       0.70      1.00      0.82        21
           2       1.00      0.57      0.73        14

    accuracy                           0.83        54
   macro avg       0.90      0.80      0.82        54
weighted avg       0.88      0.83      0.83        54

[[16  3  0]
 [ 0 21  0]
 [ 0  6  8]]


2026-05-20 11:01:13.517328: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 11:01:13.517342: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 11:01:13.517462: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmp37f9xys1
2026-05-20 11:01:13.517932: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 11:01:13.517937: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmp37f9xys1
2026-05-20 11:01:13.519117: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 11:01:13.539148: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmp37f9xys1
2026-05-

## Problem 1 - Part (c)

### Pruning

In [16]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

# <-- Enter your code here <--#
schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=(X_train.shape[0] // 8) * 10
)

In [17]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

# <-- Enter your code here <--#
model = tf.keras.Sequential([
    tf.keras.Input(shape=(num_features,)),
    tfmot.sparsity.keras.prune_low_magnitude(tf.keras.layers.Dense(64, activation='relu')),
    tfmot.sparsity.keras.prune_low_magnitude(tf.keras.layers.Dense(32, activation='relu')),
    tfmot.sparsity.keras.prune_low_magnitude(tf.keras.layers.Dense(3, activation='softmax'))
    ])

In [18]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

# <-- Enter your code here <--#
model.compile(
    optimizer=tf.keras.optimizers.legacy.Adam(),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train,
    y_train_OHE,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    callbacks=[tfmot.sparsity.keras.UpdatePruningStep()]
)

Epoch 1/10
13/13 [==============================] - 1s 7ms/step - loss: 15.8272 - accuracy: 0.2020 - val_loss: 2.7050 - val_accuracy: 0.3200
Epoch 2/10
13/13 [==============================] - 0s 2ms/step - loss: 4.3173 - accuracy: 0.2727 - val_loss: 6.9566 - val_accuracy: 0.4400
Epoch 3/10
13/13 [==============================] - 0s 1ms/step - loss: 3.2393 - accuracy: 0.3737 - val_loss: 1.1980 - val_accuracy: 0.4800
Epoch 4/10
13/13 [==============================] - 0s 1ms/step - loss: 1.4451 - accuracy: 0.4747 - val_loss: 0.9688 - val_accuracy: 0.6800
Epoch 5/10
13/13 [==============================] - 0s 1ms/step - loss: 1.2153 - accuracy: 0.5051 - val_loss: 0.8564 - val_accuracy: 0.6800
Epoch 6/10
13/13 [==============================] - 0s 1ms/step - loss: 0.8793 - accuracy: 0.6566 - val_loss: 0.8376 - val_accuracy: 0.6400
Epoch 7/10
13/13 [==============================] - 0s 1ms/step - loss: 1.3839 - accuracy: 0.5556 - val_loss: 0.7584 - val_accuracy: 0.7200
Epoch 8/10
13/13 [=

In [19]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

# <-- Enter your code here <--#
stripped_model = tfmot.sparsity.keras.strip_pruning(model)

converter = tf.lite.TFLiteConverter.from_keras_model(stripped_model)
tflite_model = converter.convert()

with open("model_pruned.tflite", "wb") as f:
    f.write(tflite_model)

file_size_kb = os.path.getsize("model_pruned.tflite") / 1024
print(f"TFLite model size: {file_size_kb:.2f} KB")

INFO:tensorflow:Assets written to: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmpm6dk59ns/assets


INFO:tensorflow:Assets written to: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmpm6dk59ns/assets


TFLite model size: 14.09 KB


2026-05-20 11:01:14.758368: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 11:01:14.758379: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 11:01:14.758469: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmpm6dk59ns
2026-05-20 11:01:14.758749: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 11:01:14.758753: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmpm6dk59ns
2026-05-20 11:01:14.759406: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 11:01:14.766862: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmpm6dk59ns
2026-05-

In [20]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#
y_pred = stripped_model.predict(X_test)

y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test_OHE, axis=1)

print(classification_report(y_true, y_pred_classes))
print(confusion_matrix(y_true, y_pred_classes))

2/2 [==============================] - 0s 854us/step
              precision    recall  f1-score   support

           0       0.46      1.00      0.63        19
           1       0.77      0.48      0.59        21
           2       0.00      0.00      0.00        14

    accuracy                           0.54        54
   macro avg       0.41      0.49      0.41        54
weighted avg       0.46      0.54      0.45        54

[[19  0  0]
 [11 10  0]
 [11  3  0]]


/Users/meghangillen/ai/projects/tinyml-arduino/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/meghangillen/ai/projects/tinyml-arduino/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/meghangillen/ai/projects/tinyml-arduino/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _war

## Problem 1 - Part (d)

### Knowledge Distillation

In [21]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)
teacher_model = tf.keras.Sequential([
    tf.keras.Input(shape=(num_features,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])
# <-- Enter your code here <--#
student_model = tf.keras.Sequential([
    tf.keras.Input(shape=(num_features,)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

In [22]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

# <-- Enter your code here <--#
y_teacher_soft = teacher_model.predict(X_train_scaled)

4/4 [==============================] - 0s 534us/step


In [23]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

# <-- Enter your code here <--#
distill_labels = np.concatenate([y_train_OHE, y_teacher_soft], axis=1)

def distillation_loss(y_true_combined, y_pred):
    y_true_hard = y_true_combined[:, :3]
    y_true_soft = y_true_combined[:, 3:]

    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)
    
    alpha = 0.5
    return alpha * hard_loss + (1 - alpha) * soft_loss
    
    # <-- Enter your code here: implement hard/soft label separation and weighted loss <--#
    #pass

In [24]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

# <-- Enter your code here <--#
student_model.compile(
    optimizer=tf.keras.optimizers.legacy.Adam(),
    loss=distillation_loss
)

history = student_model.fit(
    X_train,
    distill_labels,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
)

Epoch 1/10
13/13 [==============================] - 0s 6ms/step - loss: 71.3868 - val_loss: 30.9964
Epoch 2/10
13/13 [==============================] - 0s 1ms/step - loss: 14.5834 - val_loss: 8.8994
Epoch 3/10
13/13 [==============================] - 0s 1ms/step - loss: 8.4504 - val_loss: 4.7592
Epoch 4/10
13/13 [==============================] - 0s 1ms/step - loss: 4.6282 - val_loss: 4.4280
Epoch 5/10
13/13 [==============================] - 0s 1ms/step - loss: 3.5670 - val_loss: 3.2375
Epoch 6/10
13/13 [==============================] - 0s 1ms/step - loss: 3.3910 - val_loss: 2.9591
Epoch 7/10
13/13 [==============================] - 0s 1ms/step - loss: 2.8088 - val_loss: 2.2775
Epoch 8/10
13/13 [==============================] - 0s 1ms/step - loss: 2.2695 - val_loss: 2.0112
Epoch 9/10
13/13 [==============================] - 0s 1ms/step - loss: 1.9601 - val_loss: 1.8415
Epoch 10/10
13/13 [==============================] - 0s 1ms/step - loss: 1.8120 - val_loss: 1.7369


In [25]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

# <-- Enter your code here <--#
converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_model = converter.convert()

with open("model_kd.tflite", "wb") as f:
    f.write(tflite_model)

file_size_kb = os.path.getsize("model_kd.tflite") / 1024
print(f"Model size: {file_size_kb:.2f} KB")

INFO:tensorflow:Assets written to: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmpplibfed7/assets


INFO:tensorflow:Assets written to: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmpplibfed7/assets


Model size: 6.11 KB


2026-05-20 11:01:15.574272: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 11:01:15.574284: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 11:01:15.574371: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmpplibfed7
2026-05-20 11:01:15.574756: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 11:01:15.574760: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmpplibfed7
2026-05-20 11:01:15.575822: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 11:01:15.593149: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmpplibfed7
2026-05-

In [26]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#
y_pred = student_model.predict(X_test_scaled)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test_OHE, axis=1)

print(classification_report(y_true, y_pred_classes))
print(confusion_matrix(y_true, y_pred_classes))

2/2 [==============================] - 0s 815us/step
              precision    recall  f1-score   support

           0       0.33      0.58      0.42        19
           1       0.50      0.29      0.36        21
           2       0.56      0.36      0.43        14

    accuracy                           0.41        54
   macro avg       0.46      0.41      0.41        54
weighted avg       0.46      0.41      0.40        54

[[11  6  2]
 [13  6  2]
 [ 9  0  5]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [29]:
# <-- (if needed) Enter your code here <--#
small_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(num_features,)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

small_model.compile(
    optimizer=tf.keras.optimizers.legacy.Adam(learning_rate=0.005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = small_model.fit(
    X_train_scaled,
    y_train_OHE,
    epochs=20,
    batch_size=8,
    validation_split=0.2
)
converter = tf.lite.TFLiteConverter.from_keras_model(small_model)
tflite_model = converter.convert()

with open("small_model.tflite", "wb") as f:
    f.write(tflite_model)

file_size_kb = os.path.getsize("small_model.tflite") / 1024
print(f"Small model size: {file_size_kb:.2f} KB")

quantize_and_evaluate(small_model, X_test_scaled, y_test_OHE, 'float16', 'small_model_float16.tflite')

Epoch 1/20
13/13 [==============================] - 0s 5ms/step - loss: 0.9027 - accuracy: 0.5859 - val_loss: 0.6914 - val_accuracy: 0.7600
Epoch 2/20
13/13 [==============================] - 0s 1ms/step - loss: 0.4883 - accuracy: 0.9091 - val_loss: 0.4136 - val_accuracy: 0.8400
Epoch 3/20
13/13 [==============================] - 0s 1ms/step - loss: 0.2453 - accuracy: 0.9697 - val_loss: 0.2235 - val_accuracy: 0.9600
Epoch 4/20
13/13 [==============================] - 0s 1ms/step - loss: 0.1134 - accuracy: 0.9798 - val_loss: 0.1406 - val_accuracy: 0.9600
Epoch 5/20
13/13 [==============================] - 0s 1ms/step - loss: 0.0576 - accuracy: 0.9899 - val_loss: 0.1007 - val_accuracy: 0.9600
Epoch 6/20
13/13 [==============================] - 0s 1ms/step - loss: 0.0333 - accuracy: 1.0000 - val_loss: 0.0773 - val_accuracy: 0.9600
Epoch 7/20
13/13 [==============================] - 0s 1ms/step - loss: 0.0194 - accuracy: 1.0000 - val_loss: 0.0637 - val_accuracy: 0.9600
Epoch 8/20
13/13 [==

INFO:tensorflow:Assets written to: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmp3g3w_j4b/assets


Small model size: 6.12 KB


2026-05-20 11:06:40.280014: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 11:06:40.280037: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 11:06:40.280137: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmp3g3w_j4b
2026-05-20 11:06:40.280567: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 11:06:40.280571: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmp3g3w_j4b
2026-05-20 11:06:40.281782: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 11:06:40.300431: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmp3g3w_j4b
2026-05-

INFO:tensorflow:Assets written to: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmpbk2jzxjb/assets


INFO:tensorflow:Assets written to: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmpbk2jzxjb/assets



FLOAT16 TFLite model size: 5.02 KB
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


2026-05-20 11:06:40.539200: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 11:06:40.539214: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 11:06:40.539306: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmpbk2jzxjb
2026-05-20 11:06:40.539707: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 11:06:40.539711: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmpbk2jzxjb
2026-05-20 11:06:40.541424: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 11:06:40.561858: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/ft/hk0gywbs1zq157c1btq38ptr0000gn/T/tmpbk2jzxjb
2026-05-

# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
